# Post-Processing Visualization for SWAN Wave Model Output

---

## Overview

After running the SWAN wave model, the output fields need to be visualized for analysis and interpretation. This notebook demonstrates three standard **post-processing visualization types** commonly used in coastal engineering:

| Visualization | What it shows | Typical use |
|--------------|---------------|-------------|
| **Temporal** (time series) | Evolution of wave parameters at a single point | Identify storm events, tidal modulation, model stability |
| **Spatial** (2-D map) | Wave height distribution over the domain | Locate energy hotspots, wave shadows, refraction patterns |
| **Spectral** (polar plot) | Directional wave energy spectrum $E(f, \theta)$ | Understand wave composition (swell vs. wind-sea), spreading |

This notebook uses **synthetic data** to illustrate the visualization techniques, so it can be run independently without a SWAN installation.

### Dependencies

```
numpy, matplotlib
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
np.random.seed(42)

## 1. Temporal Visualization: Time Series of $H_s$

A time series of significant wave height at a single output point is the most basic SWAN post-processing plot. It shows how wave conditions evolve over the simulation period.

We simulate a 72-hour $H_s$ record with:
- A **diurnal variation** (sea breeze effect)
- Random noise (short-period wave variability)

In [ ]:
def plot_time_series_example():
    """Plot a 72-hour time series of significant wave height at a single point."""
    time_hours = np.arange(0, 72, 1)
    # Simulate Hs with diurnal variation + random noise
    sample_hs = 1.5 + 0.8 * np.sin(time_hours * 2 * np.pi / 24) \
                + np.random.normal(0, 0.1, len(time_hours))

    plt.figure(figsize=(10, 4))
    plt.plot(time_hours, sample_hs, label='$H_s$ (m)', color='darkblue')
    plt.title('Temporal Evolution of Significant Wave Height')
    plt.xlabel('Time (Hours Since Start)')
    plt.ylabel('Significant Wave Height ($H_s$, m)')
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.legend()
    plt.tight_layout()
    plt.show()

print("Generating temporal plot...")
plot_time_series_example()

## 2. Spatial Visualization: 2-D Map with Direction Vectors

A 2-D contour map shows the spatial distribution of wave height (or any scalar field) across the computational domain. Overlaid **quiver arrows** show the mean wave propagation direction.

We simulate a cross-shore domain where:
- $H_s$ **decreases** towards the coast due to bottom friction and breaking
- Wave direction shows slight **refraction** (bending of wave crests towards shore-parallel)

In [ ]:
def plot_spatial_map_example():
    """Plot a 2D contour map of Hs with mean wave direction quiver overlay."""
    grid_res = 50
    X = np.linspace(0, 10000, grid_res)   # Cross-shore (m)
    Y = np.linspace(0, 5000, grid_res)    # Along-shore (m)
    X_mesh, Y_mesh = np.meshgrid(X, Y)

    # Simulate shoaling: Hs decreases towards the coast (X = 10000)
    Hs_data = 3.0 * (1 - (X_mesh / 10000) * 0.5) + 0.5

    # Simulate wave direction (nautical, 'coming from') with refraction
    Dir_data_nautical = 270 - 10 * (Y_mesh / 5000)
    Dir_angle_cartesian = 270 - Dir_data_nautical

    fig, ax = plt.subplots(figsize=(10, 6))

    # Scalar field: significant wave height
    contour = ax.contourf(X_mesh, Y_mesh, Hs_data, levels=20, cmap='inferno')
    fig.colorbar(contour, label='Significant Wave Height ($H_s$, m)')

    # Vector field: mean wave direction (downsampled for clarity)
    skip = (slice(None, None, 5), slice(None, None, 5))
    U = np.cos(np.deg2rad(Dir_angle_cartesian[skip]))
    V = np.sin(np.deg2rad(Dir_angle_cartesian[skip]))
    ax.quiver(X_mesh[skip], Y_mesh[skip], U, V,
              color='cyan', pivot='mid', scale=40,
              headwidth=3, headlength=4, alpha=0.9)

    ax.set_title('Spatial Distribution of Wave Height and Direction (Refraction)')
    ax.set_xlabel('Cross-shore Distance (m)')
    ax.set_ylabel('Along-shore Distance (m)')
    plt.tight_layout()
    plt.show()

print("Generating spatial map...")
plot_spatial_map_example()

## 3. Spectral Visualization: Directional Wave Spectrum

The **directional wave energy spectrum** $E(f, \theta)$ is the most complete description of the sea state. It shows how wave energy is distributed across frequencies (related to wave period) and directions.

We simulate a spectrum with:

- **Frequency shape**: Simplified JONSWAP spectrum
  $$S(f) = f^{-5} \exp\left(-\frac{5}{4}\left(\frac{f_p}{f}\right)^4\right) \cdot \gamma^{\exp\left(-\frac{(f-f_p)^2}{2\sigma^2 f_p^2}\right)}$$
  where $f_p = 0.10$ Hz is the peak frequency and $\gamma = 3.3$ is the peak enhancement factor.

- **Directional spreading**: $\cos^{2s}$ function (Mitsuyasu et al., 1975)
  $$D(\theta) = \cos^{2s}\left(\frac{\theta - \theta_0}{2}\right)$$
  where $s = 15$ gives a narrow swell-like spreading and $\theta_0 = 270°$ (from the west).

- **Combined**: $E(f, \theta) = S(f) \cdot D(\theta)$

The polar plot uses the nautical convention (North at top, clockwise) matching SWAN's output format.

In [ ]:
def plot_spectral_example():
    """Plot a directional wave energy spectrum as a polar contour."""
    frequencies = np.linspace(0.03, 0.5, 80)      # Hz
    directions = np.linspace(0, 2 * np.pi, 90)    # radians (full circle)
    F, D = np.meshgrid(frequencies, directions)

    # Frequency spectrum: simplified JONSWAP shape
    fp = 0.10
    gamma_peak = 3.3
    sigma = np.where(F <= fp, 0.07, 0.09)
    S_f = F**(-5) * np.exp(-1.25 * (fp / F)**4) \
        * gamma_peak ** np.exp(-0.5 * ((F - fp) / (sigma * fp))**2)

    # Directional spreading: cos^2s (Mitsuyasu et al., 1975)
    theta_mean = np.deg2rad(270)     # Mean wave direction (from West)
    s = 15                           # Spreading parameter (narrow swell)
    D_theta = np.cos(0.5 * (D - theta_mean)) ** (2 * s)

    # Combined spectrum E(f, theta)
    E = S_f * D_theta
    E = E / np.nanmax(E)   # Normalize for display

    fig, ax = plt.subplots(subplot_kw={'projection': 'polar'}, figsize=(8, 8))
    contour = ax.contourf(D, F, E, levels=20, cmap='magma')
    fig.colorbar(contour, pad=0.1, label='Normalized Spectral Density')
    ax.set_theta_zero_location('N')     # North at top (nautical convention)
    ax.set_theta_direction(-1)          # Clockwise
    ax.set_title('Directional Wave Spectrum $E(f, \\theta)$', pad=20)
    ax.set_ylabel('Frequency (Hz)', labelpad=30)
    plt.tight_layout()
    plt.show()

print("Generating spectral polar plot...")
plot_spectral_example()

## Summary

This notebook demonstrated three fundamental post-processing visualizations for spectral wave model output:

| Type | Plot | Key insight |
|------|------|-------------|
| Temporal | Line plot of $H_s(t)$ | Storm timing, tidal modulation, model stability |
| Spatial | 2-D $H_s$ contour + direction quivers | Energy hotspots, refraction patterns, wave shadows |
| Spectral | Polar $E(f, \theta)$ contour | Swell vs. wind-sea, directional spreading, peak frequency |

### Applying to Real SWAN Output

To use these visualizations with actual SWAN output:

1. **Temporal**: Read TABLE output (`swan_runner.read_table()`) and plot the `Hsig` column vs. time
2. **Spatial**: Read BLOCK `.MAT` output (`swan_runner.read_mat_block()`) and plot the 2-D `Hsig` field
3. **Spectral**: Request SPECOUT in the SWAN input and read the binary spectral file

The synthetic examples above use the same plotting patterns you would apply to real model data.